In [1]:
pip install mlflow

  Using cached pandas-2.3.3-cp312-cp312-win_amd64.whl.metadata (19 kB)
  Using cached pyarrow-23.0.1-cp312-cp312-win_amd64.whl.metadata (3.1 kB)
  Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached frozenlist-1.8.0-cp312-cp312-win_amd64.whl.metadata (21 kB)
  Using cached multidict-6.7.1-cp312-cp312-win_amd64.whl.metadata (5.5 kB)
  Using cached propcache-0.4.1-cp312-cp312-win_amd64.whl.metadata (14 kB)
  Using cached blinker-1.9.0-py3-none-any.whl.metadata (1.6 kB)
  Using cached itsdangerous-2.2.0-py3-none-any.whl.metadata (1.9 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached pyasn1_modules-0.4.2-py3-none-any.whl.metadata (3.5 kB)
   ---------------------------------------- 0.0/10.5 MB ? eta -:--:--
   ------------------------------------- -- 9.7/10.5 MB 46.5 MB/s eta 0:0

  You can safely remove it manually.

[notice] A new release of pip is available: 24.2 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
import matplotlib.pyplot as plt
import pickle

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, 
                             recall_score, f1_score)
import warnings
warnings.filterwarnings('ignore')

print("✅ Librerías cargadas")

✅ Librerías cargadas


In [3]:
df = pd.read_csv("../data/dataset_preprocesado.csv")

X = df["content"]
y = df["label"]

tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1,2))
X_tfidf = tfidf.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42, stratify=y
)

print(f"✅ Datos listos — Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")

✅ Datos listos — Train: 35911 | Test: 8978


In [4]:
# Guardar experimentos en la carpeta del proyecto
mlflow.set_tracking_uri("../mlflow_runs")
mlflow.set_experiment("CleanNews_AI")

print("✅ MLflow configurado")
print("📁 Los experimentos se guardan en /mlflow_runs")

2026/05/04 20:52:04 INFO mlflow.tracking.fluent: Experiment with name 'CleanNews_AI' does not exist. Creating a new experiment.


✅ MLflow configurado
📁 Los experimentos se guardan en /mlflow_runs


In [5]:
modelos = {
    "Logistic_Regression": LogisticRegression(max_iter=1000),
    "Linear_SVM":          LinearSVC(max_iter=1000),
    "Naive_Bayes":         MultinomialNB(),
    "Random_Forest":       RandomForestClassifier(n_estimators=100, random_state=42)
}

resultados = {}

for nombre, modelo in modelos.items():
    print(f"\n⏳ Entrenando y registrando {nombre}...")
    
    with mlflow.start_run(run_name=nombre):
        # Entrenar
        modelo.fit(X_train, y_train)
        y_pred = modelo.predict(X_test)
        
        # Calcular métricas
        acc  = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred)
        rec  = recall_score(y_test, y_pred)
        f1   = f1_score(y_test, y_pred)
        
        # Registrar parámetros en MLflow
        mlflow.log_param("modelo", nombre)
        mlflow.log_param("max_features_tfidf", 10000)
        mlflow.log_param("ngram_range", "(1,2)")
        mlflow.log_param("test_size", 0.2)
        
        # Registrar métricas en MLflow
        mlflow.log_metric("accuracy",  acc)
        mlflow.log_metric("precision", prec)
        mlflow.log_metric("recall",    rec)
        mlflow.log_metric("f1_score",  f1)
        
        # Guardar el modelo en MLflow
        mlflow.sklearn.log_model(modelo, nombre)
        
        resultados[nombre] = {
            "modelo": modelo,
            "accuracy": acc,
            "precision": prec,
            "recall": rec,
            "f1": f1
        }
        
        print(f"  ✅ Accuracy: {acc:.4f} | F1: {f1:.4f}")


⏳ Entrenando y registrando Logistic_Regression...


2026/05/04 20:52:22 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/04 20:52:22 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


  ✅ Accuracy: 0.9908 | F1: 0.9903

⏳ Entrenando y registrando Linear_SVM...


2026/05/04 20:52:30 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/04 20:52:30 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/04 20:52:34 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


  ✅ Accuracy: 0.9959 | F1: 0.9957

⏳ Entrenando y registrando Naive_Bayes...


2026/05/04 20:52:34 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


  ✅ Accuracy: 0.9521 | F1: 0.9498

⏳ Entrenando y registrando Random_Forest...


2026/05/04 20:53:39 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/04 20:53:39 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


  ✅ Accuracy: 0.9979 | F1: 0.9978


In [6]:
tabla = pd.DataFrame([
    {
        "Modelo":    nombre,
        "Accuracy":  f"{d['accuracy']*100:.2f}%",
        "Precision": f"{d['precision']*100:.2f}%",
        "Recall":    f"{d['recall']*100:.2f}%",
        "F1-Score":  f"{d['f1']*100:.2f}%"
    }
    for nombre, d in resultados.items()
])

print(tabla.to_string(index=False))

             Modelo Accuracy Precision Recall F1-Score
Logistic_Regression   99.08%    98.79% 99.28%   99.03%
         Linear_SVM   99.59%    99.49% 99.65%   99.57%
        Naive_Bayes   95.21%    94.90% 95.07%   94.98%
      Random_Forest   99.79%    99.67% 99.88%   99.78%


In [7]:
mejor = max(resultados, key=lambda k: resultados[k]["f1"])
print(f"🏆 Mejor modelo: {mejor}")

# Guardar modelo y vectorizador
with open("../models/mejor_modelo.pkl", "wb") as f:
    pickle.dump(resultados[mejor]["modelo"], f)

with open("../models/tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf, f)

# Guardar nombre del mejor modelo para la API
with open("../models/mejor_modelo_nombre.txt", "w") as f:
    f.write(mejor)

print("✅ Modelo guardado en /models/mejor_modelo.pkl")
print("✅ Vectorizador guardado en /models/tfidf_vectorizer.pkl")

🏆 Mejor modelo: Random_Forest
✅ Modelo guardado en /models/mejor_modelo.pkl
✅ Vectorizador guardado en /models/tfidf_vectorizer.pkl


In [8]:
print("🌐 Para ver la interfaz visual de MLflow ejecuta en la terminal:")
print()
print("   mlflow ui --backend-store-uri ../mlflow_runs")
print()
print("   Luego abre: http://127.0.0.1:5000")

🌐 Para ver la interfaz visual de MLflow ejecuta en la terminal:

   mlflow ui --backend-store-uri ../mlflow_runs

   Luego abre: http://127.0.0.1:5000
